### Setup

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import SegformerConfig, SegformerForSemanticSegmentation

c:\Users\Hendra Wijaya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- Config

In [2]:
ROOT = Path("../../..")
RUN = ROOT / "results/02-segformer-mitb3-imagenet/v4-final"
WEIGHTS = ROOT / "model/02-segformer-mitb3-imagenet/v4-final/best.pth"
RAW = ROOT / "data/raw"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])

### Metric

In [3]:
def compute_stats(preds, targets, eps=1e-7):
    p, t = preds.float(), targets.float()
    tp = (p * t).sum(dim=(1, 2))
    fp = (p * (1 - t)).sum(dim=(1, 2))
    fn = ((1 - p) * t).sum(dim=(1, 2))
    tn = ((1 - p) * (1 - t)).sum(dim=(1, 2))
    fg = (tp + eps) / (tp + fp + fn + eps)
    bg = (tn + eps) / (tn + fp + fn + eps)
    recall = (tp + eps) / (tp + fn + eps)
    bg_acc = (tn + eps) / (tn + fp + eps)
    return {"foreground_iou": fg, 
            "background_iou": bg, 
            "miou": (fg + bg) / 2,
            "dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),
            "precision": (tp + eps) / (tp + fp + eps),
            "recall": recall, 
            "background_acc": bg_acc, 
            "mpa": (recall + bg_acc) / 2}

### Load Model

In [4]:
config = SegformerConfig.from_pretrained("nvidia/mit-b3", num_labels=2)
model = SegformerForSemanticSegmentation(config)
model.load_state_dict(torch.load(WEIGHTS, map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


### Scoring

In [5]:
rows = []

In [6]:
with torch.no_grad():
    for name in pd.read_csv(RUN / "split_test.csv")["image_name"]:
        image = cv2.cvtColor(cv2.imread(str(RAW / "Raw_Images" / name)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(RAW / "Segmentation_Masks" / name.replace(".jpg", "_mask.png")), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (512, 512))
        mask = cv2.resize((mask > 127).astype(np.uint8), (512, 512), interpolation=cv2.INTER_NEAREST)

        x = torch.from_numpy(((image / 255 - MEAN) / STD).transpose(2, 0, 1)).float()[None].to(DEVICE)
        logits = F.interpolate(model(x).logits, size=(512, 512), mode="bilinear", align_corners=False)
        pred = logits.softmax(1)[:, 1] > 0.5
        gt = torch.from_numpy(mask)[None].to(DEVICE)

        row = {k: v.item() for k, v in compute_stats(pred, gt).items()}
        row |= {"image_name": name, "gt_pct": 100 * gt.float().mean().item(), "pred_pct": 100 * pred.float().mean().item()}
        rows.append(row)

In [7]:
df = pd.DataFrame(rows)

df.head()

,foreground_iou,background_iou,miou,dice,precision,recall,background_acc,mpa,image_name,gt_pct,pred_pct
0,7.926670e-01,0.998175,0.895421,8.843438e-01,9.279509e-01,0.844651,0.999458,0.922054,img1681.jpg,0.820160,0.746536
1,6.578225e-01,0.923022,0.790422,7.935982e-01,6.687313e-01,0.975802,0.926422,0.951112,img1514.jpg,13.210678,19.276810
2,6.093846e-11,0.993740,0.496870,6.093846e-11,6.093846e-11,1.000000,0.993740,0.996870,img356.jpg,0.000000,0.625992
3,8.319157e-01,0.996472,0.914194,9.082467e-01,8.972876e-01,0.919477,0.997998,0.958737,img1141.jpg,1.866531,1.912689
4,6.342741e-01,0.995165,0.814719,7.762151e-01,6.515242e-01,0.959930,0.995513,0.977721,img1564.jpg,0.866318,1.276398


In [8]:
df.to_csv(RUN / "per_image_test.csv", index=False)

In [10]:
stored = json.loads((RUN / "test_metrics.json").read_text())
for key in ["foreground_iou", "background_iou", "miou", "dice", "precision", "recall", "mpa"]:
    print(f"{key:15s} now {df[key].mean():.4f} | saved {stored[key]:.4f}")

foreground_iou  now 0.6926 | saved 0.6926
background_iou  now 0.9715 | saved 0.9715
miou            now 0.8320 | saved 0.8320
dice            now 0.7839 | saved 0.7839
precision       now 0.7816 | saved 0.7816
recall          now 0.8515 | saved 0.8515
mpa             now 0.9138 | saved 0.9138


### 4.5. Hasil Evaluasi (Numbers)

In [11]:
empty = df[df.gt_pct == 0]
plastic = df[df.gt_pct > 0]
zero = plastic[plastic.foreground_iou < 1e-6]
print("Citra uji:", len(df))
print(f"Rata-rata luas plastik: ground truth {df.gt_pct.mean():.2f}% | prediksi {df.pred_pct.mean():.2f}%")
print("Mask kosong:", len(empty), "| prediksi kosong:", (empty.pred_pct == 0).sum(), "| ada piksel salah:", (empty.pred_pct > 0).sum())
print(f"IoU plastik citra tanpa plastik: {empty.foreground_iou.mean():.3f}")
print(f"Citra berplastik: {len(plastic)} | rata-rata {plastic.foreground_iou.mean():.3f} | median {plastic.foreground_iou.median():.3f}")
print("Prediksi lebih luas dari ground truth:", (plastic.pred_pct > plastic.gt_pct).sum())
print("IoU plastik 0:", len(zero), "| luas di bawah 1%:", (zero.gt_pct < 1).sum(), "| tanpa prediksi:", (zero.pred_pct == 0).sum())

Citra uji: 301
Rata-rata luas plastik: ground truth 5.37% | prediksi 6.52%
Mask kosong: 79 | prediksi kosong: 70 | ada piksel salah: 9
IoU plastik citra tanpa plastik: 0.886
Citra berplastik: 222 | rata-rata 0.624 | median 0.654
Prediksi lebih luas dari ground truth: 146
IoU plastik 0: 4 | luas di bawah 1%: 4 | tanpa prediksi: 3


### 4.5. Tabel 4.16

In [12]:
bins = pd.cut(plastic.gt_pct, [0, 1, 5, 15, 100], right=False)
print(plastic.groupby(bins, observed=True).foreground_iou.agg(["count", "mean"]).round(3))

           count   mean
gt_pct                 
[0, 1)        73  0.542
[1, 5)        74  0.617
[5, 15)       50  0.686
[15, 100)     25  0.758
